# Trend veri hattı — Colab'da çalıştırma

Bu notebook hocanın **SENG360 Search-LLMs-V1** kurulumunu (Colab + Ollama + llama3.2) kullanır, sonra repodaki `trendpipe` hattını çalıştırır.

**Önce:** *Runtime → Change runtime type → T4 GPU → Save*

Hücreleri yukarıdan aşağıya çalıştırın (*Runtime → Run all* de olur).

## 1. Ollama kurulumu (hocanın kodu + zstd)

In [ ]:
# Hocanın kodu + zstd: Ollama'nın güncel kurulum betiği arşivi açmak için zstd istiyor (Colab'da kurulu değil).
!apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Ollama sunucusunu arka planda başlat (hocanın kodu + hazır olana kadar bekleme)

In [ ]:
import subprocess, threading, time, urllib.request

def run_ollama():
    subprocess.run("OLLAMA_ORIGINS=* ollama serve", shell=True)

thread = threading.Thread(target=run_ollama, daemon=True)
thread.start()

# Hocanın kodunda sabit 5 sn bekleme var; burada sunucu yanıt verene kadar (en fazla 60 sn) bekliyoruz.
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        print("Ollama server is running successfully!")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama sunucusu 60 sn içinde başlamadı; önceki hücrenin çıktısını kontrol edin.")

## 3. Modeli indir
Hocanın notebook'undaki model `llama3.2` (3B). İlk iki denemede Türkçe yorumda ciddi hatalar yaptı
(yatay terime "yükseliyor", uydurma "satış artışı", Türkçe-İngilizce karışık). Bu yüzden varsayılanı T4'e sığan
`llama3.1:8b` yaptık (indirme ~5 GB). Hocanın modeliyle karşılaştırmak için `MODEL = "llama3.2"` yazın.

In [ ]:
MODEL = "llama3.1:8b"   # hocanın modeli: "llama3.2"
!ollama pull {MODEL}

## 4. Repoyu indir ve bağımlılıkları kur
Repo **özel (private)** ise Colab'ın soldaki 🔑 *Secrets* bölümüne `GITHUB_TOKEN` adıyla bir GitHub erişim anahtarı
ekleyin (GitHub → Settings → Developer settings → Fine-grained token, bu repo için *Contents: Read-only*) ve
"Notebook access" iznini açın. Repo herkese açıksa gerek yok. Anahtar ekrana yazdırılmaz.

In [ ]:
import os, subprocess
REPO = "github.com/Ylmazakdenz/My-study-AAI.git"
DEST = "/content/My-study-AAI"

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # secret yoksa herkese açık repo gibi dene
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"

if not os.path.exists(DEST):
    r = subprocess.run(["git", "clone", "-q", url, DEST], capture_output=True, text=True)
    print("Klonlandı." if r.returncode == 0 else "Klonlama başarısız (özel repo ise GITHUB_TOKEN ekleyin):\n"
          + r.stderr.replace(token or "@@", "***"))
else:
    subprocess.run(["git", "-C", DEST, "pull", "-q"])
    print("Güncellendi.")

%cd /content/My-study-AAI
!pip install -q -r requirements.txt

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ["OLLAMA_MODEL"] = MODEL
# Wikimedia, iletişim bilgisi olan istemcilere daha geniş istek sınırı tanır. İsterseniz e-postanızı yazın.
os.environ["CONTACT_EMAIL"] = ""

## 5. Model bağlantı testi

In [ ]:
!python -m trendpipe check-llm

## 6a. Kayıtlı örnek veriyle yorum (internetten veri çekmeden)
`data/sample/` altındaki gerçek örnek çalıştırmanın (18.11.2026) analizini modele gönderir. Veri çekmeden
model ve istem değişikliklerini denemek için kullanın.

In [ ]:
!python -m trendpipe interpret --out-dir data/sample --run latest

## 6b. (İsteğe bağlı) Baştan sona: veri çek → hesapla → yorumlat
Colab'ın IP adresleri bazı kaynaklarca engellenebilir; hata olursa veri çekmeyi kendi bilgisayarınızda yapın.

In [ ]:
!python -m trendpipe all

## 7. Son yorumu görüntüle

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
runs = sorted(list(Path("data/runs").glob("*/interpretation.md")) + list(Path("data/sample").glob("*/interpretation.md")), key=lambda p: p.stat().st_mtime)
display(Markdown(runs[-1].read_text(encoding="utf-8")) if runs else "Henüz yorum yok.")

## 8. Son çalıştırmayı indir
Sonuç klasörünü zip olarak bilgisayarınıza indirir (örnek veri olarak repoya eklemek için).

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
latest = max(Path("data/runs").glob("*/manifest.json"), key=lambda p: p.stat().st_mtime).parent
zip_path = shutil.make_archive(f"/content/run_{latest.name}", "zip", latest.parent, latest.name)
files.download(zip_path)